In [ ]:
%precision 8
import numpy
import matplotlib.pyplot as plt

Before you turn this problem in, make sure everything runs as expected. First, restart the kernel (in the menubar, select Kernel $\rightarrow$ Restart) and then run all cells (in the menubar, select Cell $\rightarrow$ Run All).

Make sure you fill in any place that says YOUR CODE HERE or "YOUR ANSWER HERE", as well as your name and collaborators below:

# HW 2: BVP Problems 1

## Question 1

We can derive a non-uniform 3-point approximation to $u''(x_i)$ with the stencil
$$
    u''(x_i) \approx \frac{2}{\Delta x_i ( \Delta x_i + \Delta x_{i+1} )} U_{i-1} - \frac{2}{\Delta x_i \Delta x_{i+1}} U_{i} + \frac{2}{\Delta x_{i+1} ( \Delta x_i + \Delta x_{i+1} )} U_{i+1}
$$
where $\Delta x_i = x_i - x_{i-1}$.

**(a)** [10] Show that the truncation error for this approximation is 
$$
    \tau_i = \frac{1}{3} (\Delta x_{i+1} - \Delta x_i) u^{(3)}(x_i) + \frac{1}{12} \left (\frac{\Delta x_i^3 + \Delta x_{i+1}^3}{\Delta x_i + \Delta x_{i+1}} \right ) u^{(4)}(x_i) + \cdots.
$$
This leads us to believe that using this approach would be only first order accurate as 
$$
    \tau_i = \mathcal{O}(\max (\Delta x_i, \Delta x_{i+1} ) )
$$
if $\Delta x_i \neq \Delta x_{i+1}$.

Solution Here

**(b)** [20] Implement a finite difference method based on the above discretization for the problem
$$
    u''(x) = e^x ~~~~ \Omega = [0, 1] \\
    u(0) = \alpha = 0 ~~~~ u(1) = \beta = 3
$$
whose true solution is
$$
    u(x) = e^x + (\sigma + 1) \times (x - 3) + \beta - e^3.
$$
Write the function so that it can take in a parameter `grid_type` that determines the type of grid to use.

In [ ]:
def grid_points(a, b, m, grid_type='uniform'):
    """Returns a set of m grid points between a and b
    
    The parameter `grid_type` determines what the distribution of 
    points is.
     - grid_type = 'uniform' - Uniform distribution of points
     - grid_type = 'right_layer' - Clustered near right boundary
     - grid_type = 'random' - Randomly distributed
     - grid_type = 'chebyshev' - Chebyshev extreme points
    """
    # Computational grid
    z = numpy.linspace(0, 1, m + 2)
    
    if grid_type.lower() == "uniform":
        x = a + (b - a) * z
    elif grid_type.lower() == "right_layer":
        x = a + (b - a) * (1.0 - (1.0 - z)**2)
    elif grid_type.lower() == "random":
        x = numpy.empty(m + 2)
        x[1:-1] = a + (b - a) * numpy.sort(numpy.random.uniform(size=(m)))
        x[0] = a
        x[-1] = b
    elif grid_type.lower() == "chebyshev":
        x = a + (b - a) * 0.5 * (1.0 + numpy.cos(numpy.pi * (1.0 - z)))
    else:
        raise ValueError("Unknown grid type.")
    
    return x

In [ ]:
def solve_nonuniform(a, b, alpha, beta, m, f, grid_type='uniform'):
    """Solve the ODE u'' = f(x)
    
    This function solves the ODE above using the grid specfied on the
    interval x \in [a, b] with boundary conditions u(a) = alpha and
    u(b) = beta.
    
    :Input:
     - *a* (float) - The beginning of the interval x \in [a, b]
     - *b* (float) - The end of the interval x \in [a, b]
     - *alpha* (float) - Value that defines the BC at x = a
     - *beta* (float) - Value that defines the BC at x = b
     - *m* (int) - Number of grid points to use
     - *f* (func) - Right hand side function defined in the ODE
     - *grid_type* (string) - Type of grid to use
     
    :Output:
     - *x* (ndarray) - Location of grid points used including boundaries
     - *U* (ndarray) - Solution array including boundary conditions (m + 2)
    """
    
    # Begin Solution Here
    return x, U

In [ ]:
a = 0.0
b = 1.0
alpha = 0.0
beta = 3.0
f = lambda x: numpy.exp(x)
u_true = lambda x: (4.0 - numpy.exp(1.0)) * x - 1.0 + numpy.exp(x)
x, U = solve_nonuniform(a, b, alpha, beta, 100, f, grid_type='uniform')
print(numpy.linalg.norm(u_true(x) - U, ord=2))
fig = plt.figure()
axes = fig.add_subplot(1, 1, 1)
axes.plot(x, U, 'bo')
axes.plot(x, u_true(x), 'k')
axes.set_xlabel("x")
axes.set_ylabel("u(x)")
axes.set_xlim((a, b))
assert(numpy.linalg.norm(u_true(x) - U, ord=2) < 1e-4)
print("Success!")

**(c)** [10] We would expect in general then that we would see only 1st order convergence if we let `grid_type = 'random'`.  Analyze the convergence of the method using each of the types of point distributions available in the function `grid_points`.  Use the $\Delta x$ computed from the uniform grid point distribution.

In [ ]:
# Problem setup
a = 0.0
b = 1.0
alpha = 0.0
beta = 3.0
f = lambda x: numpy.exp(x)
u_true = lambda x: (4.0 - numpy.exp(1.0)) * x - 1.0 + numpy.exp(x)

# Begin Solution Here

## Question 2

Consider the convection-diffusion problem defined by
$$
    u_t + au_x = \kappa u_{xx} + \psi(x)
$$
This is similar to the heat conduction problem except that we also have a convective term $a~u_x$.  You can think of $u(x,t)$ as describing the distribution of dye in a stream flowing at velocity $a$ where $\kappa$ describes how quickly the dye diffuses and $\psi(x)$ represents dye being added to the stream.

Considering the steady state solution of the problem above leads to the ODE
$$
    \epsilon u''(x) - u'(x) = f(x)
$$
where $\epsilon$ represents the relative strength of the advective and diffusive effects.

We will solve this problem with the domain and boundary conditions
$$
    \Omega = [0, 1] ~~~~ u(0)=\alpha ~~~~ u(1)=\beta ~~~~ f(x) = -1
$$
whose true solution is
$$
    u(x) = \alpha + x + (\beta - \alpha -1) \left( \frac{e^{x / \epsilon} - 1}{e^{1 / \epsilon} - 1} \right ).
$$

**(a)** [10] Discretize the boundary value problem presented using at least second order finite difference methods.  Make sure to pick the appropriate finite difference approach at the boundaries.

In [ ]:
def solve_adv_diff(alpha, beta, epsilon, m):
    """Solve the steady-state advection-diffusion problem
    
    :Input
     - *alpha* (float) - Value that defines the BC at x = 0
     - *beta* (float) - Value that defines the BC at x = 1
     - *epsilon* (float) - The ratio of the advection speed and diffusivity
     - *m* (int) - Number of grid points to use
     
    :Output:
     - *x* (ndarray) - Location of grid points used including boundaries
     - *U* (ndarray) - Solution array including boundary conditions (m + 2)
    """
    
    # Begin Solution Here
    
    return x, U

In [ ]:
alpha = 1.0
beta = 3.0
epsilon = -0.1
u_true = lambda x: alpha + x + (beta - alpha - 1.0) * (numpy.exp(x / epsilon) - 1.0) / (numpy.exp(1.0 / epsilon) - 1.0)
x, U = solve_adv_diff(alpha, beta, epsilon, 100)

fig = plt.figure()
axes = fig.add_subplot(1, 1, 1)
axes.plot(x, U, 'bo')
axes.plot(x, u_true(x), 'k')
axes.set_xlabel("x")
axes.set_ylabel("u(x)")
axes.set_xlim((0.0, 1.0))

error = numpy.linalg.norm((x[1] - x[0]) * (U - u_true(x)), ord=1)
print(error)
assert(error < 1e-4)
print("Success!")

**(b)** [10] Using your function from part (a) explore the parameter space a bit by varying $m$ and $\epsilon$.  Suggested values to explore include $m \in [10, 100]$ and $\epsilon \in [-0.01, -1.0]$.

Describe what you observe with both plots and written observations.  In particular do your solutions seem physically reasonable?  If your code has difficulties explain why it might be having issues.

In [ ]:
a = 0.0
b = 1.0
alpha = 1.0
beta = 3.0
epsilon = -0.1
u_true = lambda x: alpha + x + (beta - alpha - 1.0) * (numpy.exp(x / epsilon) - 1.0) / (numpy.exp(1.0 / epsilon) - 1.0)

# Begin Solution Here

Solution Here!

## Question 3

Consider the following BVP:
$$
    u''(x) + u(x) = 0 \quad \quad x \in \Omega = (a, b) \\
    u(a) = \alpha \quad u(b) = \beta
$$
where $L = b - a$.

**(a)** [10] What is the exact solution of the BVP?

Solution Here!

**(b)** [15] Write a function that solves this equation using a finite difference approximation.

In [ ]:
def solve_linear_pendulum(a, b, alpha, beta, m):
    """Solve u'' + u = 0 with x \in (a, b) and u(a) = \alpha, u(b) = \beta
    
    Returns the points the solution was approximated at including the boundaries and
    the solution
    """
    
    # Begin Solution Here

    return x, U

In [ ]:
a = 0.0
b = 1.0
alpha = 2.0
beta = 3.0
C1 = (alpha - beta * numpy.sin(a) / numpy.sin(b)) / (numpy.cos(a) - numpy.sin(a) / numpy.tan(b))
C2 = beta / numpy.sin(b) - C1 / numpy.tan(b)
u_true = lambda x: C1 * numpy.cos(x) + C2 * numpy.sin(x)
x, U = solve_linear_pendulum(a, b, alpha, beta, 100)
fig = plt.figure()
axes = fig.add_subplot(1, 1, 1)
axes.plot(x, u_true(x), 'k')
axes.plot(x, U, 'ob')
axes.set_xlim(a, b)
axes.set_xlabel("x")
axes.set_ylabel("u(x)")
error = numpy.linalg.norm((x[1] - x[0]) * (U - u_true(x)), ord=1)
print("Error = ", error)
assert(error < 1e-5)
print("Success!")

**(c)** [5] Let $a=0$ and $b=\pi$.  For what values of $\alpha$ and $\beta$ does this BVP have solutions?  Sketch out a family of solutions in the case of infinite numbers of solutions.

Solution Here!

In [ ]:
# Solution Here!

**(d)** [5] What happens numerically for the values $a = 0$, $b = \pi$, $\alpha = 1$, $\beta = -1$ as $\Delta x \rightarrow 0$?  Also try with $\alpha = \beta = 1$.

Solution Here

In [ ]:
# Solution Here

**(e)** [5] You might expect the linear system in part (c) to be singular since the boundary value problem is not well posed.  Clearly this does not seem to be the case as we are finding solutions.  One argument for why this is the case is that the finite discretization leads to a system that is solvable.  Let's investigate this computational by computing some quantities related to the matrix $A$ and show what their behavior is as $\Delta x \rightarrow 0$:
 - compute the eigenvalues of the matrix $A$,
 - compute the $L_2$ norm of $A^{-1}$, and
 - compute the condition number of the matrix $A$.
Comment on what you observe.

In [ ]:
# Solution Here

Solution Here